# Parkinson Speech Top-10 Feature Model Training

This notebook documents the full training workflow for the **New Training with 10 features** experiment.

Audience: students or reviewers who want to reproduce the PD speech classification pipeline.

By the end, you will have:
- Loaded the speech feature dataset.
- Cleaned numeric columns and labels.
- Checked and handled missing values with median imputation.
- Selected the top 10 features using mutual information.
- Trained Logistic Regression, SVM, and XGBoost models.
- Computed cross-validation metrics, confusion matrices, and saved model artifacts.


## Workflow Outline

1. Configure paths and imports.
2. Gather/load the dataset.
3. Clean the data and inspect class balance.
4. Handle missing values through model pipelines.
5. Select the top 10 features.
6. Train models with stratified 10-fold cross-validation.
7. Compare metrics and confusion matrices.
8. Fit final models and export artifacts.


In [1]:
from pathlib import Path
import copy
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    print(f"XGBoost is not available in this environment: {exc}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


XGBoost is not available in this environment: No module named 'xgboost'


In [2]:
# This notebook is saved inside: New Training with 10 features
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "New Training with 10 features":
    candidate = Path("C:/Users/saiku/OneDrive/Desktop/Projects/SDP_Test/New Training with 10 features")
    if candidate.exists():
        NOTEBOOK_DIR = candidate

DATA_PATH = NOTEBOOK_DIR / "pd_speech_features.csv"
PROCESSED_DIR = NOTEBOOK_DIR / "processed"
RESULTS_DIR = NOTEBOOK_DIR / "notebook_results"
MODELS_DIR = NOTEBOOK_DIR / "notebook_models"

PROCESSED_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Dataset path:", DATA_PATH)


Notebook directory: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features
Dataset path: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features\pd_speech_features.csv


## 1. Gather / Load Dataset

The dataset is expected as `pd_speech_features.csv` in this folder. If you download the dataset again, place it beside this notebook and keep the same file name.


In [3]:
raw_df = pd.read_csv(DATA_PATH)
print("Raw shape:", raw_df.shape)
raw_df.head()


Raw shape: (756, 755)


,id,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,class
0,0,1,0.85247,0.71826,0.57227,240.0,239.0,0.008064,0.000087,0.00218,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,0,1,0.76686,0.69481,0.53966,234.0,233.0,0.008258,0.000073,0.00195,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,0,1,0.85083,0.67604,0.58982,232.0,231.0,0.008340,0.000060,0.00176,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,1,0,0.41121,0.79672,0.59257,178.0,177.0,0.010858,0.000183,0.00419,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,1,0,0.32790,0.79782,0.53028,236.0,235.0,0.008162,0.002669,0.00535,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1


In [4]:
raw_df.info(max_cols=20)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Columns: 755 entries, id to class
dtypes: float64(749), int64(4), object(2)
memory usage: 4.4+ MB


## 2. Data Cleaning

The PD speech dataset usually contains an identifier column and a target column named `class`. We remove identifier-style columns, coerce feature columns to numeric values, and remove rows with missing labels.


In [5]:
def find_label_column(frame: pd.DataFrame) -> str:
    candidates = ["class", "target", "label", "status"]
    for column in candidates:
        if column in frame.columns:
            return column
    raise ValueError(f"Could not find a label column. Tried: {candidates}")

label_column = find_label_column(raw_df)
id_like_columns = [column for column in ["id", "subject", "subject_id", "participant_id"] if column in raw_df.columns]

clean_df = raw_df.copy()
clean_df = clean_df.drop(columns=id_like_columns, errors="ignore")
clean_df[label_column] = pd.to_numeric(clean_df[label_column], errors="coerce")
clean_df = clean_df.dropna(subset=[label_column]).copy()
clean_df[label_column] = clean_df[label_column].astype(int)

feature_columns = [column for column in clean_df.columns if column != label_column]
for column in feature_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

clean_df = clean_df.rename(columns={label_column: "target"})
feature_columns = [column for column in clean_df.columns if column != "target"]

print("Dropped ID-like columns:", id_like_columns)
print("Clean shape:", clean_df.shape)
print("Feature count:", len(feature_columns))
clean_df.head()


Dropped ID-like columns:

 ['id']
Clean shape: (756, 754)
Feature count: 753


,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,locAbsJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,target
0,1,0.85247,0.71826,0.57227,240.0,239.0,0.008064,0.000087,0.00218,0.000018,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,1,0.76686,0.69481,0.53966,234.0,233.0,0.008258,0.000073,0.00195,0.000016,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,1,0.85083,0.67604,0.58982,232.0,231.0,0.008340,0.000060,0.00176,0.000015,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,0,0.41121,0.79672,0.59257,178.0,177.0,0.010858,0.000183,0.00419,0.000046,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,0,0.32790,0.79782,0.53028,236.0,235.0,0.008162,0.002669,0.00535,0.000044,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1


In [6]:
class_balance = clean_df["target"].value_counts().sort_index().rename_axis("class").reset_index(name="count")
class_balance["percentage"] = (class_balance["count"] / class_balance["count"].sum() * 100).round(2)
class_balance


,class,count,percentage
0,0,192,25.4
1,1,564,74.6


## 3. Missing Value Inspection and Handling

We do **not** fill the entire dataset before cross-validation because that can leak information from validation folds into training folds. Instead, each model uses `SimpleImputer(strategy="median")` as the first pipeline step. During each fold, the imputer learns medians from the training split only.


In [7]:
missing_summary = (
    clean_df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)
missing_summary["missing_percentage"] = (missing_summary["missing_count"] / len(clean_df) * 100).round(3)

print("Total missing feature values:", int(missing_summary["missing_count"].sum()))
missing_summary.head(15)


Total missing feature values: 887


,feature,missing_count,missing_percentage
0,det_LT_TKEO_std_6_coef,534,70.635
1,det_entropy_log_8_coef,2,0.265
2,std_9th_delta_delta,2,0.265
3,std_12th_delta,2,0.265
4,std_delta_delta_log_energy,2,0.265
5,std_delta_delta_0th,2,0.265
6,std_1st_delta_delta,2,0.265
7,std_2nd_delta_delta,2,0.265
8,std_3rd_delta_delta,2,0.265
9,std_4th_delta_delta,2,0.265


## Save Dataset After EDA and Median Imputation

After the initial EDA checks, save two processed checkpoints:
- `eda_processed_dataset.csv`: cleaned data before imputation, kept for auditability.
- `median_imputed_dataset.csv`: cleaned data after filling missing feature values with median imputation.

The median-imputed dataset is the one used for feature selection and model training below.


In [8]:
eda_processed_dataset_path = PROCESSED_DIR / "eda_processed_dataset.csv"
median_imputed_dataset_path = PROCESSED_DIR / "median_imputed_dataset.csv"
eda_class_balance_path = PROCESSED_DIR / "eda_class_balance.csv"
eda_missing_summary_path = PROCESSED_DIR / "eda_missing_summary.csv"
imputation_values_path = PROCESSED_DIR / "median_imputation_values.csv"

# Save the cleaned post-EDA data before imputation for auditability.
clean_df.to_csv(eda_processed_dataset_path, index=False)

# Fill missing feature values using the median strategy.
median_imputer = SimpleImputer(strategy="median")
imputed_features = pd.DataFrame(
    median_imputer.fit_transform(clean_df[feature_columns]),
    columns=feature_columns,
    index=clean_df.index,
)
median_imputed_df = imputed_features.copy()
median_imputed_df["target"] = clean_df["target"].to_numpy(dtype=int)
median_imputed_df.to_csv(median_imputed_dataset_path, index=False)

imputation_values = pd.DataFrame({
    "feature": feature_columns,
    "median_value_used_for_imputation": median_imputer.statistics_,
})
imputation_values.to_csv(imputation_values_path, index=False)

class_balance.to_csv(eda_class_balance_path, index=False)
missing_summary.to_csv(eda_missing_summary_path, index=False)

print("Saved cleaned post-EDA dataset:", eda_processed_dataset_path)
print("Saved median-imputed dataset:", median_imputed_dataset_path)
print("Saved imputation values:", imputation_values_path)
print("Remaining missing values after median imputation:", int(median_imputed_df.isna().sum().sum()))


Saved cleaned post-EDA dataset: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features\processed\eda_processed_dataset.csv
Saved median-imputed dataset: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features\processed\median_imputed_dataset.csv
Saved imputation values: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features\processed\median_imputation_values.csv
Remaining missing values after median imputation: 0


## 4. Feature Selection: Top 10 by Mutual Information

Mutual information estimates how much each feature helps explain the class label. This notebook uses the saved median-imputed dataset for feature ranking and model training.


In [9]:
X_all = median_imputed_df[feature_columns]
y = median_imputed_df["target"].to_numpy(dtype=int)

# The dataset has already been median-imputed above, so feature ranking can run directly.
mi_scores = mutual_info_classif(X_all, y, random_state=RANDOM_SEED)
feature_ranking = (
    pd.DataFrame({"feature": feature_columns, "mutual_info_score": mi_scores})
    .sort_values("mutual_info_score", ascending=False)
    .reset_index(drop=True)
)
feature_ranking.insert(0, "rank", np.arange(1, len(feature_ranking) + 1))

top10_features = feature_ranking.head(10)["feature"].tolist()
print("Top 10 features:")
for index, feature in enumerate(top10_features, start=1):
    print(f"{index}. {feature}")

feature_ranking.head(10)


Top 10 features:
1. tqwt_entropy_log_dec_35
2. std_delta_delta_log_energy
3. std_8th_delta_delta
4. mean_MFCC_2nd_coef
5. tqwt_TKEO_mean_dec_16
6. tqwt_entropy_shannon_dec_35
7. tqwt_TKEO_std_dec_12
8. tqwt_maxValue_dec_12
9. tqwt_entropy_log_dec_11
10. tqwt_TKEO_mean_dec_12


,rank,feature,mutual_info_score
0,1,tqwt_entropy_log_dec_35,0.107912
1,2,std_delta_delta_log_energy,0.107570
2,3,std_8th_delta_delta,0.097704
3,4,mean_MFCC_2nd_coef,0.096951
4,5,tqwt_TKEO_mean_dec_16,0.095531
5,6,tqwt_entropy_shannon_dec_35,0.094599
6,7,tqwt_TKEO_std_dec_12,0.093492
7,8,tqwt_maxValue_dec_12,0.092444
8,9,tqwt_entropy_log_dec_11,0.092258
9,10,tqwt_TKEO_mean_dec_12,0.091247


In [10]:
X_top10 = median_imputed_df[top10_features]

feature_schema = pd.DataFrame({"position": range(len(top10_features)), "feature_name": top10_features})
feature_schema.to_csv(PROCESSED_DIR / "feature_schema_from_notebook.csv", index=False)
feature_ranking.head(10).to_csv(PROCESSED_DIR / "final_top10_features_from_notebook.csv", index=False)
median_imputed_df.to_csv(PROCESSED_DIR / "cleaned_features_from_notebook.csv", index=False)

print("Saved feature-selection processed outputs to:", PROCESSED_DIR)


Saved feature-selection processed outputs to: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features\processed


## 5. Model Training Setup

The classical models below are trained with stratified 10-fold cross-validation. Pipelines are used so every fold applies the same preprocessing order:

`median imputation -> scaling if needed -> classifier`


In [11]:
def build_models() -> dict[str, Pipeline]:
    models = {
        "LogisticRegression": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
            ]
        ),
        "SVM": Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_SEED)),
            ]
        ),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("model", XGBClassifier(
                    n_estimators=250,
                    max_depth=3,
                    learning_rate=0.05,
                    subsample=0.85,
                    colsample_bytree=0.85,
                    eval_metric="logloss",
                    random_state=RANDOM_SEED,
                )),
            ]
        )
    return models

models = build_models()
models


{'LogisticRegression': Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                 ('scaler', StandardScaler()),
                 ('model',
                  LogisticRegression(class_weight='balanced', max_iter=2000,
                                     random_state=42))]),
 'SVM': Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                 ('scaler', StandardScaler()),
                 ('model',
                  SVC(C=10.0, class_weight='balanced', probability=True,
                      random_state=42))])}

In [12]:
def predict_scores(model: Pipeline, X_valid: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_valid)[:, 1]
    if hasattr(model, "decision_function"):
        decision = model.decision_function(X_valid)
        return 1.0 / (1.0 + np.exp(-decision))
    return model.predict(X_valid).astype(float)


def evaluate_models(X: pd.DataFrame, y: np.ndarray, models: dict[str, Pipeline], folds: int = 10):
    splitter = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_SEED)
    fold_rows = []
    oof_rows = []
    confusion_rows = []

    for model_name, base_model in models.items():
        oof_pred = np.zeros(len(y), dtype=int)
        oof_score = np.zeros(len(y), dtype=float)

        for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y), start=1):
            model = copy.deepcopy(base_model)
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]

            model.fit(X_train, y_train)
            pred = model.predict(X_valid).astype(int)
            score = predict_scores(model, X_valid)

            oof_pred[valid_idx] = pred
            oof_score[valid_idx] = score

            fold_rows.append({
                "model_name": model_name,
                "fold": fold,
                "accuracy": accuracy_score(y_valid, pred),
                "recall": recall_score(y_valid, pred, zero_division=0),
                "f1": f1_score(y_valid, pred, zero_division=0),
                "roc_auc": roc_auc_score(y_valid, score),
                "fit_sample_count": len(train_idx),
            })

            for row_idx, truth, row_pred, row_score in zip(valid_idx, y_valid, pred, score):
                oof_rows.append({
                    "row_index": int(row_idx),
                    "model_name": model_name,
                    "fold": fold,
                    "y_true": int(truth),
                    "y_pred": int(row_pred),
                    "y_score": float(row_score),
                })

        matrix = confusion_matrix(y, oof_pred, labels=[0, 1])
        for actual_idx, actual_label in enumerate([0, 1]):
            for predicted_idx, predicted_label in enumerate([0, 1]):
                confusion_rows.append({
                    "model_name": model_name,
                    "actual_label": actual_label,
                    "predicted_label": predicted_label,
                    "count": int(matrix[actual_idx, predicted_idx]),
                })

    return pd.DataFrame(fold_rows), pd.DataFrame(oof_rows), pd.DataFrame(confusion_rows)


## 6. Run Cross-Validation

This cell trains and evaluates every model. It may take a little while because each model is fit once per fold.


In [13]:
fold_metrics, oof_predictions, confusion_matrices = evaluate_models(X_top10, y, models, folds=10)

model_comparison = (
    fold_metrics
    .groupby("model_name", as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_fit_sample_count=("fit_sample_count", "mean"),
    )
    .sort_values("mean_f1", ascending=False)
)

model_comparison


,model_name,mean_accuracy,mean_recall,mean_f1,mean_roc_auc,mean_fit_sample_count
1,SVM,0.824053,0.854574,0.878299,0.882519,680.4
0,LogisticRegression,0.731386,0.726817,0.800774,0.850594,680.4


In [14]:
fold_metrics.to_csv(RESULTS_DIR / "cv_fold_metrics.csv", index=False)
oof_predictions.to_csv(RESULTS_DIR / "oof_predictions.csv", index=False)
confusion_matrices.to_csv(RESULTS_DIR / "confusion_matrix.csv", index=False)
model_comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

print("Saved metrics to:", RESULTS_DIR)


Saved metrics to: C:\Users\saiku\OneDrive\Desktop\Projects\SDP_Test\New Training with 10 features\notebook_results


In [15]:
confusion_matrices.pivot_table(
    index=["model_name", "actual_label"],
    columns="predicted_label",
    values="count",
    fill_value=0,
)


predicted_label                      0      1
model_name         actual_label              
LogisticRegression 0             143.0   49.0
                   1             154.0  410.0
SVM                0             141.0   51.0
                   1              82.0  482.0

## 7. Fit Final Models and Save Artifacts

After cross-validation, final inference-ready models are fit on the full top-10 feature dataset. The saved `.joblib` files include preprocessing and the trained classifier in one pipeline.


In [16]:
artifact_rows = []
for model_name, model in models.items():
    model.fit(X_top10, y)
    artifact_path = MODELS_DIR / f"{model_name.lower()}.joblib"
    joblib.dump(model, artifact_path)
    artifact_rows.append({
        "model_name": model_name,
        "artifact_path": str(artifact_path),
        "feature_schema_path": str(PROCESSED_DIR / "feature_schema_from_notebook.csv"),
        "selected_feature_count": len(top10_features),
        "missing_value_strategy": "SimpleImputer(strategy='median') inside Pipeline",
        "inference_ready": True,
    })

artifact_manifest = pd.DataFrame(artifact_rows)
artifact_manifest.to_csv(NOTEBOOK_DIR / "notebook_model_manifest.csv", index=False)
artifact_manifest


,model_name,artifact_path,feature_schema_path,selected_feature_count,missing_value_strategy,inference_ready
0,LogisticRegression,C:\Users\saiku\OneDrive\Desktop\Projects\SDP_T...,C:\Users\saiku\OneDrive\Desktop\Projects\SDP_T...,10,SimpleImputer(strategy='median') inside Pipeline,True
1,SVM,C:\Users\saiku\OneDrive\Desktop\Projects\SDP_T...,C:\Users\saiku\OneDrive\Desktop\Projects\SDP_T...,10,SimpleImputer(strategy='median') inside Pipeline,True


## 8. Validation Checks

These checks confirm that the notebook handled missing values and exported the expected files.


In [17]:
print("Missing values in selected median-imputed top-10 features:", int(X_top10.isna().sum().sum()))

for model_name, model in models.items():
    step_names = [name for name, _ in model.steps]
    print(f"{model_name}: {step_names}")
    assert step_names[0] == "imputer"

expected_files = [
    RESULTS_DIR / "model_comparison.csv",
    RESULTS_DIR / "cv_fold_metrics.csv",
    MODELS_DIR / "svm.joblib" if "SVM" in models else None,
    NOTEBOOK_DIR / "notebook_model_manifest.csv",
]
for file_path in [item for item in expected_files if item is not None]:
    print(file_path.name, "exists=", file_path.exists())


Missing values in selected median-imputed top-10 features: 0
LogisticRegression: ['imputer', 'scaler', 'model']
SVM: ['imputer', 'scaler', 'model']
model_comparison.csv exists= True
cv_fold_metrics.csv exists= True
svm.joblib exists= True
notebook_model_manifest.csv exists= True


## Exercise

Try changing `top10_features = feature_ranking.head(10)` to `head(15)` and rerun the model section. Compare whether the extra features improve mean F1 or ROC-AUC.

Common pitfall: do not call `fit_transform` on the full dataset before cross-validation for final model metrics. Keep imputation inside the pipeline so the validation fold remains unseen during preprocessing.
